In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""Bình phương tối thiểu cho hồi quy tuyến tính đa biến với 2 biến độc lập dùng tổng + quy tắc Cramer.

Mô hình:
    y = b + a1*x1 + a2*x2

Chương trình thực hiện đúng theo giả mã:
- Tính các tổng:
    sum_x1, sum_x2, sum_y,
    sum_x1_2, sum_x2_2,
    sum_x1x2, sum_x1y, sum_x2y

- Tính các định thức:
    D, D_a1, D_a2

- Tính hệ số:
    a1 = D_a1 / D
    a2 = D_a2 / D
    b = (sum_y - a1*sum_x1 - a2*sum_x2)/n

Không sử dụng numpy/pandas/sklearn.
"""

from typing import List, Tuple


def least_squares_2var(
        data: List[Tuple[float, float, float]],
        verbose: bool = True):
    """
    Khớp mô hình:
        y = b + a1*x1 + a2*x2

    bằng quy tắc Cramer trên hệ phương trình chuẩn.

    Tham số:
        data: danh sách (x1, x2, y)
        verbose: in các bước tính trung gian

    Trả về:
        (a1, a2, b)
    """

    n = len(data)

    if n == 0:
        raise ValueError("Tập dữ liệu rỗng")

    # 1) Khởi tạo các tổng
    sum_x1 = 0.0
    sum_x2 = 0.0
    sum_y = 0.0
    sum_x1_2 = 0.0
    sum_x2_2 = 0.0
    sum_x1x2 = 0.0
    sum_x1y = 0.0
    sum_x2y = 0.0

    # Duyệt dữ liệu và cộng dồn
    for (x1, x2, y) in data:

        x1 = float(x1)
        x2 = float(x2)
        y = float(y)

        sum_x1 += x1
        sum_x2 += x2
        sum_y += y

        sum_x1_2 += x1 * x1
        sum_x2_2 += x2 * x2

        sum_x1x2 += x1 * x2

        sum_x1y += x1 * y
        sum_x2y += x2 * y

    if verbose:
        print("===== CÁC GIÁ TRỊ TỔNG =====")
        print(f"n          = {n}")
        print(f"sum_x1     = {sum_x1}")
        print(f"sum_x2     = {sum_x2}")
        print(f"sum_y      = {sum_y}")
        print(f"sum_x1_2   = {sum_x1_2}")
        print(f"sum_x2_2   = {sum_x2_2}")
        print(f"sum_x1x2   = {sum_x1x2}")
        print(f"sum_x1y    = {sum_x1y}")
        print(f"sum_x2y    = {sum_x2y}")

    # 2) Tính định thức D của ma trận hệ số
    #
    # Hệ phương trình chuẩn:
    #
    # [ n        sum_x1     sum_x2   ] [ b  ]   [ sum_y   ]
    # [ sum_x1   sum_x1_2   sum_x1x2 ] [ a1 ] = [ sum_x1y ]
    # [ sum_x2   sum_x1x2   sum_x2_2 ] [ a2 ]   [ sum_x2y ]

    D = (
        n * (sum_x1_2 * sum_x2_2 - sum_x1x2**2)
        - sum_x1 * (
            sum_x1 * sum_x2_2
            - sum_x2 * sum_x1x2
        )
        + sum_x2 * (
            sum_x1 * sum_x1x2
            - sum_x2 * sum_x1_2
        )
    )

    # 3) Tính định thức cho a1

    D_a1 = (
        n * (
            sum_x1y * sum_x2_2
            - sum_x2y * sum_x1x2
        )
        - sum_y * (
            sum_x1 * sum_x2_2
            - sum_x2 * sum_x1x2
        )
        + sum_x2 * (
            sum_x1 * sum_x2y
            - sum_x2 * sum_x1y
        )
    )

    # Tính định thức cho a2

    D_a2 = (
        n * (
            sum_x1_2 * sum_x2y
            - sum_x1x2 * sum_x1y
        )
        - sum_x1 * (
            sum_x1 * sum_x2y
            - sum_x2 * sum_x1y
        )
        + sum_y * (
            sum_x1 * sum_x1x2
            - sum_x2 * sum_x1_2
        )
    )

    if verbose:
        print("\n===== ĐỊNH THỨC =====")
        print(f"D     = {D}")
        print(f"D_a1  = {D_a1}")
        print(f"D_a2  = {D_a2}")

    # Kiểm tra hệ có nghiệm duy nhất không
    if abs(D) < 1e-12:
        raise ValueError(
            "Định thức D bằng 0 hoặc quá nhỏ -> hệ suy biến"
        )

    # 4) Tính các hệ số

    a1 = D_a1 / D
    a2 = D_a2 / D

    b = (
        sum_y
        - a1 * sum_x1
        - a2 * sum_x2
    ) / n

    if verbose:

        print("\n===== HỆ SỐ =====")

        print(f"a1 = {a1}")
        print(f"a2 = {a2}")
        print(f"b  = {b}")

        print("\nMô hình:")
        print("y_hat = b + a1*x1 + a2*x2")

        print(
            f"y_hat = {b:.10f}"
            f" + {a1:.10f}*x1"
            f" + {a2:.10f}*x2"
        )

    return a1, a2, b


if __name__ == "__main__":

    # Dữ liệu mẫu
    data = [
        (15, 105, 1.87),
        (30, 105, 2.02),
        (60, 105, 3.28),
        (15, 120, 3.05),
        (30, 120, 4.07),
        (60, 120, 5.54),
        (15, 135, 5.03),
        (30, 135, 6.45),
        (60, 135, 7.26),
    ]

    least_squares_2var(data, verbose=True)

===== CÁC GIÁ TRỊ TỔNG =====
n          = 9
sum_x1     = 315.0
sum_x2     = 1080.0
sum_y      = 38.57
sum_x1_2   = 14175.0
sum_x2_2   = 130950.0
sum_x1x2   = 37800.0
sum_x1y    = 1490.25
sum_x2y    = 4801.950000000001

===== ĐỊNH THỨC =====
D     = 38272500.0
D_a1  = 1704644.999999985
D_a2  = 4920142.50000006

===== HỆ SỐ =====
a1 = 0.04453968253968215
a2 = 0.12855555555555712
b  = -12.700000000000173

Mô hình:
y_hat = b + a1*x1 + a2*x2
y_hat = -12.7000000000 + 0.0445396825*x1 + 0.1285555556*x2
